In [5]:
from pathlib import Path

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "Raw Data").is_dir()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not find a project directory containing 'Raw Data'.")

raw_data_folder = project_root / "Raw Data"
file_count = sum(path.is_file() for path in raw_data_folder.rglob("*"))

print(f"Raw data folder: {raw_data_folder.resolve()}")
print(f"Number of files: {file_count}")

Raw data folder: /Users/farlos/University/Perovskites-Research/Raw Data
Number of files: 134


In [6]:
import re
from collections import defaultdict
from pathlib import Path

raw_data_folder = project_root / "Raw Data"  # ใช้ project_root จากเซลล์เดิมใน paper_debug.ipynb


def paper_key(path: Path) -> str:
    """ตัด _SampleN ออก แล้วเหลือเฉพาะตัวอักษร/ตัวเลขพิมพ์เล็ก
    10.1002_solr.202400534 และ 10_1002_solr_202400534 -> 101002solr202400534
    10.1002admi.202500231 และ 10.1002_admi.202500231   -> 101002admi202500231
    """
    stem = re.sub(r"_Sample\d+$", "", path.stem, flags=re.I)
    return re.sub(r"[^0-9a-z]", "", stem.lower())


papers = defaultdict(list)
for f in raw_data_folder.glob("*.xlsx"):
    if f.name.startswith("~$"):  # ไฟล์ล็อกชั่วคราวของ Excel
        continue
    papers[paper_key(f)].append(f.name)

print(f"จำนวน paper: {len(papers)}  (จากไฟล์ {sum(map(len, papers.values()))} ไฟล์)")

# ดูว่ามี DOI ไหนถูกเขียนหลายแบบ แล้วถูกรวมเป็นเปเปอร์เดียว
for key, names in papers.items():
    spellings = {re.sub(r"_Sample\d+\.xlsx$", "", n, flags=re.I) for n in names}
    if len(spellings) > 1:
        print(key, sorted(spellings))


จำนวน paper: 43  (จากไฟล์ 134 ไฟล์)
101002solr202400534 ['10.1002_solr.202400534', '10_1002_solr_202400534']
101021acsami5c16972 ['10.1021_acsami.5c16972', '10_1021_acsami_5c16972']
